# 9. Dashboard data export

| | |
|---|---|
| **Stage** | 9 of 9 |
| **Purpose** | Build the data files read by the Streamlit dashboard from the final predictions, and assemble every generated file of the dashboard repository in a fresh folder, de-identified and checked, as a single zip. |
| **Inputs** | `outputs/predictions_v4_ens/{game}_pairs.parquet`<br>`outputs/predictions/{game}_aspects.parquet`<br>`outputs/dashboard/overall_{game}.parquet`<br>`data/processed/{game}_clean.csv`<br>`outputs/final_v4/*.csv, figures/*.png, run_manifest.json`<br>`outputs/label_audit/label_audit_summary.csv`<br>`scripts/aspect_taxonomy.py` |
| **Outputs** | `outputs/dashboard_v4/ (reviews_meta.parquet, pairs.parquet, examples.parquet, trend.csv, ranking_v2_vs_v4.csv)`<br>`github_staging/dashboard_data/ and github_staging/dashboard_data.zip (data/ and docs/figures/)`<br>`github_staging/dashboard_data_manifest.csv` |
| **Runtime** | CPU only. |
| **Depends on** | Stages 4, 7 and 8 |
| **Configuration** | Set `PROJECT_ROOT` in the first code cell if your Drive folder is not `MyDrive/GameSense`. |

## What this notebook does

It runs on CPU, from top to bottom, and produces **one zip with every generated file of the dashboard repository**:
1. **Step 1** builds the dashboard data pack in `outputs/dashboard_v4/` (four files), scrubbing personal-data patterns from the review text they contain.
2. **Step 2** copies all generated files into a **fresh** staging folder (an old one is deleted first, so no stale file can survive), de-identifies and checks the staged copies, and downloads `dashboard_data.zip`.

The zip holds only generated results (`data/` and `docs/figures/`). The code and documentation of the repository (`app.py`, `README.md`, `docs/`, `notebooks/`, `src/`) are maintained in the repository itself.

**Safety.** Original files are only read. Step 1 writes only to `outputs/dashboard_v4/`; step 2 writes only to `github_staging/`.

## Step 1 · Build the dashboard data pack

Reads the v4 ensemble predictions and the cleaned reviews and writes `reviews_meta.parquet`, `pairs.parquet`, `examples.parquet` and `trend.csv`.

**Check the output for**
- the `de-identification` lines and the line saying that no email, link, @handle or long-number pattern remains;
- `pairs: 34800` (a NOTE appears otherwise);
- `at_null`, `version_null` and `overall_null` close to 0 (a missing version is normal for some games);
- the line `no personal-data columns in any output file`.

In [ ]:
# ===== Dashboard data pack (v4): built from the v4 ENSEMBLE predictions (CPU only) =====
# Reads predictions and cleaned reviews; writes ONLY to outputs/dashboard_v4/ (older dashboard folders are left untouched).
import os, sys, glob
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

PROJECT_ROOT = "/content/drive/MyDrive/GameSense"
sys.path.insert(0, f"{PROJECT_ROOT}/scripts")
from aspect_taxonomy import ASPECT_IDS
GAME_SLUGS = ["royalmatch", "genshin", "mlbb", "pubgm", "arknights"]

PRED_DIR = f"{PROJECT_ROOT}/outputs/predictions_v4_ens"      # the prediction set shown in the dashboard
PRED_V2 = f"{PROJECT_ROOT}/outputs/predictions"              # read-only: sentence counts per review + v2 ranking comparison
OVERALL_DIR = f"{PROJECT_ROOT}/outputs/dashboard"            # read-only: Task 1 per-review sentiment (overall_<game>.parquet)
OUT_DASH = f"{PROJECT_ROOT}/outputs/dashboard_v4"
os.makedirs(OUT_DASH, exist_ok=True)
_OUT = os.path.abspath(OUT_DASH)
def safe_path(p):                                            # this cell may write ONLY inside outputs/dashboard_v4
    q = os.path.abspath(p)
    assert q == _OUT or q.startswith(_OUT + os.sep), f"refusing to write outside dashboard_v4: {q}"
    return q

WINDOW = {"arknights": 91}                                   # days per trend window (sparse game); all other games use 14
CAUTION_ASPECTS = {"advertisement", "account_support"}       # examples for these come only from 1-2 star reviews (conservative)
MAX_SEED_STD = 0.20                                          # examples must be stable across the ensemble seeds
MIN_MENTIONS = 30
COMMON = ["performance_stability", "monetization_value", "advertisement", "level_difficulty_balance",
          "content_progression", "paywall", "account_support"]
GENRE = {"royalmatch": [], "genshin": ["gacha_rates", "character_operator_balance"],
         "mlbb": ["character_operator_balance", "matchmaking", "cheating"], "pubgm": ["cheating", "matchmaking"],
         "arknights": ["gacha_rates", "character_operator_balance", "grind_progression"]}

def parse_robust(s):
    try:
        t = pd.to_datetime(s, format="mixed", utc=True, errors="coerce")
    except (ValueError, TypeError):
        from dateutil import parser
        t = pd.to_datetime(s.map(lambda x: parser.parse(x) if isinstance(x, str) and x.strip() else pd.NaT), errors="coerce", utc=True)
    return t.dt.tz_localize(None)

PAIR_COLS = ["sentence_uid", "reviewId", "game", "sentence_text", "aspect", "p_aspect", "sentiment", "p_negative", "p_neg_std"]
metas, pairs_all, trends, examples = [], [], [], []
for slug in GAME_SLUGS:
    clean = pd.read_csv(f"{PROJECT_ROOT}/data/processed/{slug}_clean.csv", dtype={"at": str},
                        usecols=["reviewId", "at", "score", "reviewCreatedVersion", "content_clean", "thumbsUpCount", "market"]).drop_duplicates("reviewId")
    clean["at"] = parse_robust(clean["at"])
    clean["version"] = clean["reviewCreatedVersion"].astype("string").str.split("_").str[0]      # Genshin: "6.7.0_4548.._4576.." -> "6.7.0"
    ov = pd.read_parquet(f"{OVERALL_DIR}/overall_{slug}.parquet").drop_duplicates("reviewId")[["reviewId", "overall"]]
    sents = pd.read_parquet(f"{PRED_V2}/{slug}_aspects.parquet", columns=["reviewId"])
    meta = clean.merge(ov, on="reviewId", how="left")
    meta["game"] = slug
    meta["n_sentences"] = meta["reviewId"].map(sents.groupby("reviewId").size()).fillna(0).astype(int)
    meta["thumbs"] = pd.to_numeric(meta["thumbsUpCount"], errors="coerce").fillna(0)

    avail = pd.read_parquet(f"{PRED_DIR}/{slug}_pairs.parquet").columns
    pairs = pd.read_parquet(f"{PRED_DIR}/{slug}_pairs.parquet", columns=[c for c in PAIR_COLS if c in avail])
    if "reviewId" not in pairs.columns:                                                          # sentence_uid = reviewId + "_" + index
        pairs["reviewId"] = pairs["sentence_uid"].astype(str).str.rsplit("_", n=1).str[0]
    if "game" not in pairs.columns:
        pairs["game"] = slug
    if "p_aspect" not in pairs.columns:
        print(f"WARNING {slug}: p_aspect missing -> example ranking uses sentiment confidence only"); pairs["p_aspect"] = 1.0
    pairs = pairs.rename(columns={"p_neg_std": "p_negative_seed_std"})                           # name used by the dashboard
    if "p_negative_seed_std" not in pairs.columns:
        pairs["p_negative_seed_std"] = np.nan
    pairs = pairs.merge(meta[["reviewId", "at", "version", "score", "thumbs", "content_clean", "market"]], on="reviewId", how="left")
    pairs = pairs.rename(columns={"score": "stars"})

    # ---- trend table: fixed windows; rates per 100 sentences; all markets and US-only; overall and by star band ----
    wd = WINDOW.get(slug, 14)
    t0 = meta["at"].min().normalize()
    meta["win"] = ((meta["at"] - t0).dt.days // wd).astype("Int64")
    meta["band"] = pd.cut(pd.to_numeric(meta["score"], errors="coerce"), [0, 2, 3, 5], labels=["low", "mid", "high"])
    asp = [a for a in ASPECT_IDS if a in (COMMON + GENRE[slug]) and a != "grind_progression"]
    wins = np.arange(int(meta["win"].max()) + 1)
    for scope in ["all", "us"]:
        ms = meta if scope == "all" else meta[meta["market"] == "us"]
        pw = pairs.merge(ms[["reviewId", "win", "band"]], on="reviewId", how="inner")
        tot = ms.groupby("win").agg(n_reviews=("reviewId", "size"), n_sent=("n_sentences", "sum")).reindex(wins, fill_value=0)
        totb = (ms.groupby(["win", "band"], observed=True)["n_sentences"].sum().unstack(fill_value=0)
                .reindex(index=wins, columns=["low", "mid", "high"], fill_value=0))
        def cnt(df):
            return df.groupby(["win", "aspect"]).size().unstack(fill_value=0).reindex(index=wins, columns=asp, fill_value=0)
        men, neg = cnt(pw), cnt(pw[pw["sentiment"] == "negative"])
        men_low, men_high = cnt(pw[pw["band"] == "low"]), cnt(pw[pw["band"] == "high"])
        for a in asp:
            trends.append(pd.DataFrame({
                "game": slug, "market_scope": scope, "aspect": a, "window_start": [t0 + pd.Timedelta(days=int(w) * wd) for w in wins],
                "window_days": wd, "n_reviews": tot["n_reviews"].values, "n_sentences": tot["n_sent"].values,
                "mentions": men[a].values, "negative_mentions": neg[a].values,
                "mentions_per100": 100 * men[a].values / np.where(tot["n_sent"].values > 0, tot["n_sent"].values, np.nan),
                "negative_per100": 100 * neg[a].values / np.where(tot["n_sent"].values > 0, tot["n_sent"].values, np.nan),
                "mentions_per100_1to2star": 100 * men_low[a].values / np.where(totb["low"].values > 0, totb["low"].values, np.nan),
                "mentions_per100_4to5star": 100 * men_high[a].values / np.where(totb["high"].values > 0, totb["high"].values, np.nan),
                "reliable_window": tot["n_reviews"].values >= 100}))

    # ---- representative negative examples ----
    ex = pairs[pairs["sentiment"] == "negative"].copy()
    ex = ex[~(ex["aspect"].isin(CAUTION_ASPECTS) & ~(ex["stars"] <= 2))]
    ex = ex[ex["p_negative_seed_std"].fillna(0) <= MAX_SEED_STD]
    ex["rank_score"] = ex["p_aspect"] * ex["p_negative"]
    ex = ex.sort_values(["aspect", "rank_score", "thumbs"], ascending=[True, False, False]).groupby("aspect").head(30)
    examples.append(ex[["game", "aspect", "sentence_text", "content_clean", "stars", "p_aspect", "p_negative", "p_negative_seed_std",
                        "rank_score", "thumbs", "version", "at", "market"]])
    metas.append(meta[["game", "reviewId", "at", "version", "score", "overall", "n_sentences", "thumbs", "market"]])
    pairs_all.append(pairs[["game", "sentence_uid", "reviewId", "aspect", "p_aspect", "sentiment", "p_negative", "p_negative_seed_std",
                            "stars", "at", "version", "market", "sentence_text"]])
    print(f"{slug:11s} reviews {len(meta):6d} | pairs {len(pairs):6d} | examples {len(ex):4d}")

meta_all, pairs_df = pd.concat(metas, ignore_index=True), pd.concat(pairs_all, ignore_index=True)
trend_df, ex_df = pd.concat(trends, ignore_index=True), pd.concat(examples, ignore_index=True)
# ---- de-identification of the review text shown in the dashboard: emails, links, @handles, numbers of 8+ digits (possible game ids), phone-like numbers ----
import re
PII = [("email", "[EMAIL]", re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")), ("url", "[URL]", re.compile(r"https?://\S+|www\.\S+")),
       ("handle", "[HANDLE]", re.compile(r"(?<![\w@])@\w{3,}")), ("long_number", "[NUMBER]", re.compile(r"(?<!\d)\d{8,}(?!\d)")),
       ("phone_like", "[PHONE]", re.compile(r"(?<!\d)\+?\d[\d\s().-]{7,}\d(?!\d)"))]
def scrub(series):
    s_ = series.astype("string")
    for _, ph, rx in PII:
        s_ = s_.str.replace(rx, ph, regex=True)
    return s_
for nm, df_, cols in [("pairs", pairs_df, ["sentence_text"]), ("examples", ex_df, ["sentence_text", "content_clean"])]:
    for c_ in cols:
        new = scrub(df_[c_]); changed = int((new.fillna("") != df_[c_].astype("string").fillna("")).sum()); df_[c_] = new
        print(f"de-identification: {nm}.{c_}: {changed} rows changed")

for name, df in [("reviews_meta.parquet", meta_all), ("pairs.parquet", pairs_df), ("examples.parquet", ex_df)]:
    df.to_parquet(safe_path(f"{OUT_DASH}/{name}"), index=False)
trend_df.to_csv(safe_path(f"{OUT_DASH}/trend.csv"), index=False)

# ---- checks ----
print("\nFiles in", OUT_DASH)
for f in sorted(os.listdir(OUT_DASH)):
    print(f"  {f:24s} {os.path.getsize(f'{OUT_DASH}/{f}') / 1024 / 1024:6.2f} MB")
chk = meta_all.groupby("game").agg(reviews=("reviewId", "size"), first=("at", "min"), last=("at", "max"),
                                   at_null=("at", lambda s: round(100 * s.isna().mean(), 1)),
                                   version_null=("version", lambda s: round(100 * s.isna().mean(), 1)),
                                   market_null=("market", lambda s: round(100 * s.isna().mean(), 1)),
                                   overall_null=("overall", lambda s: round(100 * s.isna().mean(), 1)))
print("\n", chk.to_string())
print("\npairs:", len(pairs_df), "| with missing date:", int(pairs_df["at"].isna().sum()), "| with seed std:", int(pairs_df["p_negative_seed_std"].notna().sum()),
      "| trend rows:", len(trend_df), "| example rows:", len(ex_df))
if len(pairs_df) != 34800:
    print(f"NOTE: {len(pairs_df)} pairs (the full v4 run has 34,800) - check that DRY_RUN was False when predictions_v4_ens was written")
for nm, df in [("reviews_meta", meta_all), ("pairs", pairs_df), ("examples", ex_df)]:
    bad = [c for c in df.columns if c.lower() in ("username", "user_name", "userimage", "user_image")]
    assert not bad, f"{nm} contains personal-data columns: {bad}"
print("no personal-data columns in any output file")
for nm, df in [("pairs", pairs_df), ("examples", ex_df)]:
    left = {n: int(df["sentence_text"].astype("string").str.contains(rx).fillna(False).sum()) for n, _, rx in PII}
    assert sum(left.values()) == 0, f"{nm}: personal-data patterns remain in sentence_text: {left}"
print("no email, link, @handle or long-number pattern remains in the dashboard text")

# ---- does the model change alter the ISSUE RANKING? (priority = negative mentions per 100 sentences) ----
rows = []
for slug in GAME_SLUGS:
    S = meta_all.loc[meta_all["game"] == slug, "n_sentences"].sum()
    v2 = pd.read_parquet(f"{PRED_V2}/{slug}_pairs.parquet", columns=["aspect", "sentiment", "review_score"])
    v4 = pairs_df[pairs_df["game"] == slug][["aspect", "sentiment", "stars"]].rename(columns={"stars": "review_score"})
    def prio(p, validated=False):
        q = p[p["sentiment"] == "negative"]
        if validated:                                    # conservative view used with v2: ads/account negatives only from 1-2 star reviews
            q = q[~q["aspect"].isin(CAUTION_ASPECTS) | (q["review_score"] <= 2)]
        return 100 * q.groupby("aspect").size() / S
    t = pd.DataFrame({"mentions": v4.groupby("aspect").size(), "v2_raw": prio(v2), "v2_validated": prio(v2, True), "v4_ens": prio(v4)}).fillna(0)
    t = t[(t["mentions"] >= MIN_MENTIONS) & (t.index != "grind_progression")]
    for c in ["v2_raw", "v2_validated", "v4_ens"]:
        t[f"rank_{c}"] = t[c].rank(ascending=False, method="first").astype(int)
    t.insert(0, "game", slug)
    rows.append(t.reset_index().rename(columns={"index": "aspect"}))
rk = pd.concat(rows, ignore_index=True)
rk.round(3).to_csv(safe_path(f"{OUT_DASH}/ranking_v2_vs_v4.csv"), index=False)
print("\n=== Spearman between v4 ensemble ranking and v2 rankings ===")
for slug in GAME_SLUGS:
    t = rk[rk["game"] == slug]
    print(f"{slug:11s} vs v2 raw {t['v4_ens'].rank().corr(t['v2_raw'].rank()):.2f} | vs v2 conservative {t['v4_ens'].rank().corr(t['v2_validated'].rank()):.2f}")


## Step 2 · Assemble, de-identify, check and zip

The cell
1. checks that every required source exists (it stops with a clear message otherwise);
2. rebuilds the staging folder `github_staging/dashboard_data/` from scratch;
3. scrubs the text columns of **every** staged table, including the error-example tables, and verifies that no user-name column and no personal-data pattern is left;
4. stops without writing a zip if a file is missing, still contains personal data, or is larger than the 25 MB web-upload limit;
5. writes `dashboard_data_manifest.csv` (size, rows, checksum; kept outside the zip) and downloads `dashboard_data.zip`.

In [ ]:
# ===== Step 2: build the dashboard repository's generated files in a FRESH folder, check them, and download one zip =====
# Originals are only read. The staging folder is deleted and rebuilt on every run, so no old file can survive in it.
import os, re, shutil, hashlib
import numpy as np, pandas as pd
import pyarrow.parquet as pq
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

PROJECT_ROOT = "/content/drive/MyDrive/GameSense"
FINAL = f"{PROJECT_ROOT}/outputs/final_v4"
PACK = f"{PROJECT_ROOT}/outputs/dashboard_v4"
STAGING_PARENT = f"{PROJECT_ROOT}/github_staging"
STAGING = f"{STAGING_PARENT}/dashboard_data"                 # becomes the contents of the zip: data/ and docs/figures/
ZIP_PATH = f"{STAGING_PARENT}/dashboard_data.zip"
WEB_LIMIT_MB = 25                                             # the GitHub web uploader refuses larger single files

# (source file, destination inside the repository, required?)
PLAN = [
    (f"{PACK}/reviews_meta.parquet", "data/reviews_meta.parquet", True),
    (f"{PACK}/pairs.parquet", "data/pairs.parquet", True),
    (f"{PACK}/examples.parquet", "data/examples.parquet", True),
    (f"{PACK}/trend.csv", "data/trend.csv", True),
    (f"{FINAL}/06_issue_priority_by_game.csv", "data/issue_priority_by_game.csv", True),
    (f"{FINAL}/06_cross_game_common_priority.csv", "data/cross_game_common_priority.csv", True),
    (f"{FINAL}/06_cross_game_common_neg_share.csv", "data/cross_game_common_neg_share.csv", False),
    (f"{FINAL}/06_priority_robustness_1to2.csv", "data/priority_robustness_1to2.csv", False),
    (f"{FINAL}/07_version_windows_and_stars.csv", "data/version_windows_and_stars.csv", True),
    (f"{FINAL}/07_version_aspect_changes.csv", "data/version_aspect_changes.csv", False),
    (f"{FINAL}/07b_version_boundary_sensitivity.csv", "data/version_boundary_sensitivity.csv", False),
    (f"{FINAL}/07b_market_composition.csv", "data/market_composition.csv", False),
    (f"{FINAL}/07b_nonUS_share_by_window.csv", "data/nonUS_share_by_window.csv", False),
    (f"{FINAL}/02_task3_overall.csv", "data/task3_overall.csv", True),
    (f"{FINAL}/02_task3_by_aspect.csv", "data/task3_by_aspect.csv", True),
    (f"{FINAL}/03_task2_overall.csv", "data/task2_overall.csv", False),
    (f"{FINAL}/03_task2_by_aspect.csv", "data/task2_by_aspect.csv", True),
    (f"{FINAL}/08_proposal_check.csv", "data/proposal_check.csv", False),
    (f"{FINAL}/run_manifest.json", "data/run_manifest.json", False),
    (f"{PROJECT_ROOT}/outputs/label_audit/label_audit_summary.csv", "data/label_audit_summary.csv", False),   # summary numbers only, no sentences
    (f"{FINAL}/figures/issue_priority.png", "docs/figures/issue_priority.png", False),
    (f"{FINAL}/figures/task3_confusion.png", "docs/figures/task3_confusion.png", False),
]
for n09 in ["baselines_task2", "baselines_task3", "task1_overall", "task1_confusion", "task1_by_star", "task1_by_game", "task1_error_by_marker", "task1_error_examples",
            "task2_by_game", "task2_by_aspect_all_models", "task2_confusable_aspects", "task2_error_by_marker", "task2_error_examples", "task3_error_by_marker", "task3_error_examples"]:
    PLAN.append((f"{FINAL}/09_{n09}.csv", f"data/{n09}.csv", False))

# de-identification patterns (same as the data pack and notebook 10)
PII = [("email", "[EMAIL]", re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")), ("url", "[URL]", re.compile(r"https?://\S+|www\.\S+")),
       ("handle", "[HANDLE]", re.compile(r"(?<![\w@])@\w{3,}")), ("long_number", "[NUMBER]", re.compile(r"(?<!\d)\d{8,}(?!\d)")),
       ("phone_like", "[PHONE]", re.compile(r"(?<!\d)\+?\d[\d\s().-]{7,}\d(?!\d)"))]
TEXT_COLS = {"content", "content_clean", "replyContent", "sentence_text", "review_text", "text"}
USER_COLS = {"userName", "userImage"}
def scrub(series):
    s = series.astype("string")
    for _, ph, rx in PII: s = s.str.replace(rx, ph, regex=True)
    return s
def remaining(series):
    s = series.astype("string"); return sum(int(s.str.contains(rx).fillna(False).sum()) for _, _, rx in PII)

# ---------------------------------------------------------------- 1. are all sources there?
rows = [{"repo_path": d, "exists": os.path.exists(s), "required": r, "size_MB": round(os.path.getsize(s) / 1024 / 1024, 2) if os.path.exists(s) else None} for s, d, r in PLAN]
plan = pd.DataFrame(rows)
missing_required = plan[(~plan["exists"]) & plan["required"]]
print(f"{int(plan['exists'].sum())} of {len(plan)} planned files exist.")
if len(plan[~plan["exists"]]): print("not found:", ", ".join(plan.loc[~plan["exists"], "repo_path"]))
assert missing_required.empty, f"required files are missing: {missing_required['repo_path'].tolist()}. Run step 1 and notebook 07 first."

# ---------------------------------------------------------------- 2. fresh staging folder and copy
assert os.path.abspath(STAGING).startswith(os.path.abspath(STAGING_PARENT) + os.sep), "staging folder must be inside github_staging"
if os.path.exists(STAGING): shutil.rmtree(STAGING)
os.makedirs(STAGING)
for src, dst, _ in PLAN:
    if os.path.exists(src):
        p = os.path.join(STAGING, dst); os.makedirs(os.path.dirname(p), exist_ok=True); shutil.copy2(src, p)

# ---------------------------------------------------------------- 3. de-identify and check the STAGED copies
report, problems = [], []
for src, dst, _ in PLAN:
    p = os.path.join(STAGING, dst)
    if not os.path.exists(p): continue
    size = os.path.getsize(p) / 1024 / 1024; note, changed, n_rows = [], 0, ""
    if dst.endswith((".csv", ".parquet")):
        df = pd.read_csv(p, low_memory=False) if dst.endswith(".csv") else pd.read_parquet(p)
        n_rows = len(df)
        if USER_COLS & set(df.columns): problems.append(f"{dst}: contains {sorted(USER_COLS & set(df.columns))}")
        for c in [c for c in df.columns if c in TEXT_COLS]:
            new = scrub(df[c]); ch = int((new.fillna("") != df[c].astype("string").fillna("")).sum())
            if ch: df[c] = new; changed += ch
        if changed:                                              # rewrite the staged copy only if something was replaced
            df.to_csv(p, index=False) if dst.endswith(".csv") else df.to_parquet(p, index=False)
            note.append(f"{changed} text value(s) de-identified")
        left = sum(remaining(df[c]) for c in df.columns if c in TEXT_COLS)
        if left: problems.append(f"{dst}: {left} personal-data pattern(s) still present")
    if size > WEB_LIMIT_MB: problems.append(f"{dst}: {size:.1f} MB exceeds the {WEB_LIMIT_MB} MB web-upload limit")
    report.append({"repo_path": dst, "size_MB": round(os.path.getsize(p) / 1024 / 1024, 2), "rows": n_rows, "notes": "; ".join(note) or "ok"})
rep = pd.DataFrame(report)
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 60)
print("\n" + rep.to_string(index=False))
print(f"\n{len(rep)} files, {rep['size_MB'].sum():.1f} MB")
assert not problems, "STOPPED, nothing was zipped:\n - " + "\n - ".join(problems)
print("All checks passed: no user-name column, no personal-data pattern, every file within the web-upload limit.")

# ---------------------------------------------------------------- 4. manifest (outside the zip) and the zip
def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()[:12]
rep["sha256_12"] = [sha(os.path.join(STAGING, r)) for r in rep["repo_path"]]
rep.to_csv(f"{STAGING_PARENT}/dashboard_data_manifest.csv", index=False)
if os.path.exists(ZIP_PATH): os.remove(ZIP_PATH)
shutil.make_archive(ZIP_PATH[:-4], "zip", root_dir=STAGING)
print(f"\nzip: {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1024 / 1024:.1f} MB)\nmanifest: {STAGING_PARENT}/dashboard_data_manifest.csv")
try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception:
    print("(not running in Colab: take the zip from the path above)")
print("\nNext: unzip it, then in the dashboard repository use Add file > Upload files and drag in the folders 'data' and 'docs' (they replace the old files of the same name).")


## Step 3 · Upload to the dashboard repository

1. Unzip `dashboard_data.zip`. It contains two folders, `data/` and `docs/`.
2. In the repository choose **Add file > Upload files** and drag in the two folders. Files with the same name are replaced; new files are added.
3. Single files larger than 25 MB cannot be uploaded in the browser (the cell checks this).
4. After the commit, the dashboard redeploys; open it in a private window to check the result.